# T4 matcher notebook-check

Pure-logic probe for `email_parser.matcher.match` (spec §2). For each scenario it prints the intermediate sets a wrong result would hide — normalized company, candidate ids, role tokens, survivor ids — then confirms the shipped `match()` returns the expected id. No LLM, no I/O.

In [1]:
import json
from pathlib import Path
from email_parser.matcher import match
from email_parser.normalize import normalize
from email_parser.models import ExtractedFields

JOBS = json.loads(Path('tests/fixtures/jobs_snapshot.json').read_text())
for j in JOBS:
    print(j['id'], '| company=', repr(j['company']), '| title=', repr(j['title']))

job-1 | company= 'Acme' | title= 'Senior Backend Engineer'
job-2 | company= 'Acme' | title= 'Data Scientist'
job-3 | company= 'Globex' | title= 'ML Engineer'
job-4 | company= None | title= 'Recruiter (agency)'
job-5 | company= 'Initech' | title= None


In [2]:
def trace(record, jobs):
    """Mirror spec §2 step by step, printing each intermediate set."""
    ck = normalize(record.company_raw)
    print('  company_raw=', repr(record.company_raw), '-> normalized=', repr(ck))
    if ck == '':
        print('  -> empty company: return None (jobs never inspected)')
        return None
    cands = [j for j in jobs if normalize(j.get('company')) == ck and normalize(j.get('company')) != '']
    print('  candidates=', [j['id'] for j in cands])
    if len(cands) == 0:
        print('  -> 0 candidates: None'); return None
    if len(cands) == 1:
        print('  -> 1 candidate: link', cands[0]['id']); return cands[0]['id']
    rk = normalize(record.role_raw)
    print('  >=2 candidates; role_raw=', repr(record.role_raw), '-> normalized=', repr(rk))
    if rk == '':
        print('  -> no role text: None (never guess)'); return None
    toks = set(rk.split())
    print('  role_tokens=', toks)
    survivors = []
    for j in cands:
        jt = set(normalize(j.get('title')).split())
        overlap = jt & toks
        print('    ', j['id'], 'title_tokens=', jt, '| overlap=', overlap or '{}')
        if overlap:
            survivors.append(j)
    print('  survivors=', [j['id'] for j in survivors])
    if len(survivors) == 1:
        print('  -> unique survivor: link', survivors[0]['id']); return survivors[0]['id']
    print('  -> zero/ambiguous survivors: None'); return None

def rec(company_raw=None, role_raw=None):
    return ExtractedFields(company_raw=company_raw, role_raw=role_raw)

In [3]:
print('Scenario 1: unambiguous single match links')
_r = rec(company_raw='globex')
_jobs = JOBS
_traced = trace(_r, _jobs)
_actual = match(_r, _jobs)
_expected = 'job-3'
print('  shipped match() =', repr(_actual), '| expected', repr(_expected))
assert _traced == _actual, 'trace disagrees with shipped match()'
print('  PASS' if _actual == _expected else '  FAIL')

Scenario 1: unambiguous single match links
  company_raw= 'globex' -> normalized= 'globex'
  candidates= ['job-3']
  -> 1 candidate: link job-3
  shipped match() = 'job-3' | expected 'job-3'
  PASS


In [4]:
print('Scenario 2: no candidate company stays unlinked')
_r = rec(company_raw='Nonesuch')
_jobs = JOBS
_traced = trace(_r, _jobs)
_actual = match(_r, _jobs)
_expected = None
print('  shipped match() =', repr(_actual), '| expected', repr(_expected))
assert _traced == _actual, 'trace disagrees with shipped match()'
print('  PASS' if _actual == _expected else '  FAIL')

Scenario 2: no candidate company stays unlinked
  company_raw= 'Nonesuch' -> normalized= 'nonesuch'
  candidates= []
  -> 0 candidates: None
  shipped match() = None | expected None
  PASS


In [5]:
print('Scenario 3: empty company_raw stays unlinked')
_r = rec(company_raw='')
_jobs = JOBS
_traced = trace(_r, _jobs)
_actual = match(_r, _jobs)
_expected = None
print('  shipped match() =', repr(_actual), '| expected', repr(_expected))
assert _traced == _actual, 'trace disagrees with shipped match()'
print('  PASS' if _actual == _expected else '  FAIL')

Scenario 3: empty company_raw stays unlinked
  company_raw= '' -> normalized= ''
  -> empty company: return None (jobs never inspected)
  shipped match() = None | expected None
  PASS


In [6]:
print('Scenario 4: two roles resolved by role token')
_r = rec(company_raw='Acme', role_raw='backend engineer')
_jobs = JOBS
_traced = trace(_r, _jobs)
_actual = match(_r, _jobs)
_expected = 'job-1'
print('  shipped match() =', repr(_actual), '| expected', repr(_expected))
assert _traced == _actual, 'trace disagrees with shipped match()'
print('  PASS' if _actual == _expected else '  FAIL')

Scenario 4: two roles resolved by role token
  company_raw= 'Acme' -> normalized= 'acme'
  candidates= ['job-1', 'job-2']
  >=2 candidates; role_raw= 'backend engineer' -> normalized= 'backend engineer'
  role_tokens= {'engineer', 'backend'}
     job-1 title_tokens= {'engineer', 'backend', 'senior'} | overlap= {'engineer', 'backend'}
     job-2 title_tokens= {'scientist', 'data'} | overlap= {}
  survivors= ['job-1']
  -> unique survivor: link job-1
  shipped match() = 'job-1' | expected 'job-1'
  PASS


In [7]:
print('Scenario 5: two roles, no role text stays unlinked')
_r = rec(company_raw='Acme', role_raw=None)
_jobs = JOBS
_traced = trace(_r, _jobs)
_actual = match(_r, _jobs)
_expected = None
print('  shipped match() =', repr(_actual), '| expected', repr(_expected))
assert _traced == _actual, 'trace disagrees with shipped match()'
print('  PASS' if _actual == _expected else '  FAIL')

Scenario 5: two roles, no role text stays unlinked
  company_raw= 'Acme' -> normalized= 'acme'
  candidates= ['job-1', 'job-2']
  >=2 candidates; role_raw= None -> normalized= ''
  -> no role text: None (never guess)
  shipped match() = None | expected None
  PASS


In [8]:
print('Scenario 6: two roles, non-overlapping role stays unlinked')
_r = rec(company_raw='Acme', role_raw='Product Manager')
_jobs = JOBS
_traced = trace(_r, _jobs)
_actual = match(_r, _jobs)
_expected = None
print('  shipped match() =', repr(_actual), '| expected', repr(_expected))
assert _traced == _actual, 'trace disagrees with shipped match()'
print('  PASS' if _actual == _expected else '  FAIL')

Scenario 6: two roles, non-overlapping role stays unlinked
  company_raw= 'Acme' -> normalized= 'acme'
  candidates= ['job-1', 'job-2']
  >=2 candidates; role_raw= 'Product Manager' -> normalized= 'product manager'
  role_tokens= {'product', 'manager'}
     job-1 title_tokens= {'engineer', 'backend', 'senior'} | overlap= {}
     job-2 title_tokens= {'scientist', 'data'} | overlap= {}
  survivors= []
  -> zero/ambiguous survivors: None
  shipped match() = None | expected None
  PASS


In [9]:
print('Scenario 7: null title never wins the tiebreak')
_r = rec(company_raw='Acme', role_raw='backend')
_jobs = [{'id':'acme-null','company':'Acme','title':None},{'id':'acme-be','company':'Acme','title':'Backend Engineer'}]
_traced = trace(_r, _jobs)
_actual = match(_r, _jobs)
_expected = 'acme-be'
print('  shipped match() =', repr(_actual), '| expected', repr(_expected))
assert _traced == _actual, 'trace disagrees with shipped match()'
print('  PASS' if _actual == _expected else '  FAIL')

Scenario 7: null title never wins the tiebreak
  company_raw= 'Acme' -> normalized= 'acme'
  candidates= ['acme-null', 'acme-be']
  >=2 candidates; role_raw= 'backend' -> normalized= 'backend'
  role_tokens= {'backend'}
     acme-null title_tokens= set() | overlap= {}
     acme-be title_tokens= {'engineer', 'backend'} | overlap= {'backend'}
  survivors= ['acme-be']
  -> unique survivor: link acme-be
  shipped match() = 'acme-be' | expected 'acme-be'
  PASS


In [10]:
print('Scenario 8: never raises on null fields')
_r = rec()
_jobs = JOBS
_traced = trace(_r, _jobs)
_actual = match(_r, _jobs)
_expected = None
print('  shipped match() =', repr(_actual), '| expected', repr(_expected))
assert _traced == _actual, 'trace disagrees with shipped match()'
print('  PASS' if _actual == _expected else '  FAIL')

Scenario 8: never raises on null fields
  company_raw= None -> normalized= ''
  -> empty company: return None (jobs never inspected)
  shipped match() = None | expected None
  PASS
